# 01 - AgentCore Foundations

In this notebook you will establish the shared foundation for the module:

1. Place AgentCore Runtime, Observability, and Evaluations in one mental model.
2. Inspect a deterministic Strands agent and its AgentCore CLI project.
3. Test locally, deploy once, and invoke the deployed runtime.
4. Follow one session through traces and spans.

Later notebooks reuse this deployment. Do not redeploy unless you change the agent.

**Estimated time:** 35-50 minutes  
**Creates AWS resources:** Yes

## 1. The service map

| Service or tool | Question it answers |
|---|---|
| AgentCore Runtime | Where does my agent code run? |
| AgentCore Observability | What happened during an invocation? |
| AgentCore Evaluations | How well did the session, response, or tool call perform? |
| AgentCore CLI | How do I develop, deploy, inspect, and evaluate the project? |
| Python SDK | How do I automate evaluations and curated scenario runs? |

A useful hierarchy is:

```text
session
  +-- trace: one user turn and agent response
        +-- model spans
        +-- tool-call spans
        +-- application spans
```

Evaluators target one of three levels:

- `SESSION`: the whole conversation or task
- `TRACE`: one agent turn
- `TOOL_CALL`: one instrumented tool execution

Evaluators inspect instrumented activity. They do not receive a model's private hidden reasoning.

## 2. Runtime or Harness?

AgentCore offers two agent-loop choices:

- **Runtime** hosts an agent loop that you wrote. It supports multiple frameworks and protocols.
- **Harness** is a managed, configuration-driven loop for model, tool, skill, and memory composition.

This module uses Runtime with the HTTP protocol because seeing the agent, tool, and telemetry boundaries makes the evaluation examples concrete. The same evaluation concepts also apply to Harness traces.

## 3. Install notebook dependencies

Run this notebook from `Framework Specific Evaluations/AgentCore/`.

The workshop uses the Node.js package `@aws/agentcore`. Before continuing, verify that `agentcore --help` includes commands such as `create`, `dev`, `deploy`, `traces`, and `run eval`.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

MODULE_ROOT = Path.cwd()
assert (MODULE_ROOT / "agentcore" / "agentcore.json").exists(), (
    "Open this notebook from the AgentCore module directory."
)

print("Python:", sys.version.split()[0])
subprocess.run(["node", "--version"], check=True)
subprocess.run(["agentcore", "--help"], check=True)
subprocess.run(["aws", "sts", "get-caller-identity"], check=True)

## 4. Inspect the project before deploying

AgentCore CLI projects are schema-first:

- `agentcore/agentcore.json` describes deployable resources.
- `agentcore/aws-targets.json` records account and region targets.
- `app/CityAnalyst/` contains the Runtime application.
- `agentcore/.cli/deployed-state.json` is generated deployment state.

This workshop uses a `CodeZip` build, so Docker is not required. The runtime is authenticated with AWS IAM, has OpenTelemetry enabled, and uses a bounded session lifetime.

In [ ]:
config = json.loads(
    (MODULE_ROOT / "agentcore" / "agentcore.json").read_text()
)
config["runtimes"][0]

## 5. Why the agent is deterministic

Live web search produces changing results and demographic values, which makes it unsuitable for stable evaluation ground truth.

`CityAnalyst` uses a fixed fixture and three tools:

| Tool | Intended use |
|---|---|
| `lookup_city(city, state)` | One named city |
| `compare_cities(city_a, state_a, city_b, state_b)` | A two-city comparison |
| `calculate_density(population, land_area_mi2)` | Math from user-supplied values |

The agent returns structured JSON. This gives us stable facts, meaningful tool choices, checkable parameters, and a deterministic schema failure mode.

In [ ]:
facts = json.loads(
    (MODULE_ROOT / "app" / "CityAnalyst" / "data" / "city_facts.json").read_text()
)
print(facts["data_version"])
print(f"Cities: {len(facts['cities'])}")
facts["cities"][:3]

In [ ]:
print(
    (MODULE_ROOT / "app" / "CityAnalyst" / "system_prompt.txt").read_text()
)

## 6. Validate and test locally

First validate the project:

In [ ]:
subprocess.run(["agentcore", "validate"], cwd=MODULE_ROOT, check=True)

Use two terminals for the fastest local loop:

**Terminal 1**

```bash
agentcore dev --runtime CityAnalyst --no-browser
```

**Terminal 2**

```bash
agentcore dev --runtime CityAnalyst               "Compare Seattle, WA with Portland, OR." --stream
```

Local development emits OpenTelemetry by default when AWS credentials are available. Add `--no-traces` only when you intentionally do not want telemetry.

For durable multi-turn memory across cold starts or processes, use AgentCore Memory. The workshop agent keeps a small, bounded in-process session cache only so the multi-turn evaluation exercise remains focused.

## 7. Enable trace search

CloudWatch Transaction Search is a one-time account prerequisite for searchable AgentCore traces. Enable it in the CloudWatch console or run:

```bash
aws xray update-trace-segment-destination --destination CloudWatchLogs
```

Trace ingestion is asynchronous. We will poll for the session instead of sleeping for an arbitrary number of minutes.

## 8. Deploy the shared runtime

The first `agentcore deploy` establishes a deployment target and creates the CDK-managed resources. Review the target account and region shown by the CLI before confirming.

Workshop convenience settings such as public network mode and automatically generated IAM roles are not a production security baseline. Production runtimes should use least-privilege roles, an appropriate VPC posture, KMS-encrypted logs, retention settings, and an explicit inbound authorizer.

In [ ]:
subprocess.run(["agentcore", "deploy"], cwd=MODULE_ROOT, check=True)

In [ ]:
subprocess.run(["agentcore", "status"], cwd=MODULE_ROOT, check=True)

## 9. Invoke one traceable session

Use an explicit session ID so the invocation, trace, and evaluation can be connected later.

In [ ]:
from src.workshop_utils import (
    GENERATED_DIR,
    RUNTIME_NAME,
    make_session_id,
    run_cli_json,
)

GENERATED_DIR.mkdir(exist_ok=True)
SESSION_ID = make_session_id("foundations")
PROMPT = "Compare Seattle, WA with Portland, OR. Which city is denser?"

invocation = run_cli_json(
    "invoke",
    "--runtime",
    RUNTIME_NAME,
    "--session-id",
    SESSION_ID,
    "--prompt",
    PROMPT,
)
print(json.dumps(invocation, indent=2))

session_record = {
    "session_id": SESSION_ID,
    "prompt": PROMPT,
}
(GENERATED_DIR / "foundation_session.json").write_text(
    json.dumps(session_record, indent=2) + "\n"
)

## 10. Poll for the trace

The polling helper calls `agentcore traces list --json` until the session appears or the timeout expires.

In [ ]:
from src.workshop_utils import wait_for_session_trace

trace_summary = wait_for_session_trace(SESSION_ID)
trace_summary

In [ ]:
TRACE_ID = trace_summary["traceId"]
trace_path = GENERATED_DIR / "foundation_trace.json"
trace_download = run_cli_json(
    "traces",
    "get",
    TRACE_ID,
    "--runtime",
    RUNTIME_NAME,
    "--output",
    str(trace_path),
)
trace_download

## 11. Read the span hierarchy

A real trace is preferred. The checked-in fixture keeps this exercise usable before deployment and documents the minimum fields used by later custom-evaluator examples.

In [ ]:
import pandas as pd
from src.workshop_utils import find_span_records, summarize_spans

source_path = (
    trace_path
    if trace_path.exists()
    else MODULE_ROOT / "data" / "sample_spans.json"
)
trace_document = json.loads(source_path.read_text())
span_rows = summarize_spans(find_span_records(trace_document))
pd.DataFrame(span_rows)

## 12. Foundation checkpoint

You should now be able to explain:

- why Runtime, Observability, and Evaluations are separate layers
- how session, trace, and tool-call evaluation levels differ
- why fixed data is better than live search for regression ground truth
- how a session ID connects an invocation to its trace

Continue to [02 - Built-in On-Demand Evaluations](02-built-in-on-demand-evaluations.ipynb) to score this instrumented behavior.